# Task 3

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
class MultimodalHouseModel(nn.Module):
    def __init__(self):
        super(MultimodalHouseModel, self).__init__()

        # --- BRANCH 1: THE TABULAR MLP (For numerical data) ---
        self.tabular_branch = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        # --- BRANCH 2: THE CNN (For House Images) ---
        self.image_branch = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2), # 224 -> 112
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # 112 -> 56
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 32)
        )

        # --- THE FUSION HEAD ---
        # 32 (tab) + 32 (img) = 64 features
        self.regressor = nn.Sequential(
            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Linear(16, 1) # Single output: Predicted Price
        )

    def forward(self, x_tab, x_img):
        tab_feat = self.tabular_branch(x_tab)
        img_feat = self.image_branch(x_img)
        combined = torch.cat((tab_feat, img_feat), dim=1)
        return self.regressor(combined)

# Move model to GPU
model = MultimodalHouseModel().to('cuda')
print(model)

MultimodalHouseModel(
  (tabular_branch): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
  )
  (image_branch): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Flatten(start_dim=1, end_dim=-1)
    (8): Linear(in_features=100352, out_features=32, bias=True)
  )
  (regressor): Sequential(
    (0): Linear(in_features=64, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [3]:
class HouseDataset(Dataset):
    def __init__(self, size=200):
        self.size = size
        # Generate 10 random house features (sqft, beds, etc)
        self.tabular_data = torch.randn(size, 10)
        # Generate random house prices as targets
        self.prices = torch.randn(size, 1)
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        # Create a random RGB image to act as a 'house photo'
        img_array = (np.random.rand(224, 224, 3) * 255).astype('uint8')
        img = Image.fromarray(img_array)
        img = self.transform(img)
        return self.tabular_data[idx], img, self.prices[idx]

# Create DataLoaders
train_loader = DataLoader(HouseDataset(size=300), batch_size=16, shuffle=True)
test_loader = DataLoader(HouseDataset(size=50), batch_size=16, shuffle=False)

print("Synthetic Dataset generated and ready for training.")

Synthetic Dataset generated and ready for training.


In [4]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- Training Loop ---
model.train()
print("Starting Training...")
for epoch in range(5):
    epoch_loss = 0
    for tab, img, price in train_loader:
        tab, img, price = tab.to('cuda'), img.to('cuda'), price.to('cuda')

        optimizer.zero_grad()
        outputs = model(tab, img)
        loss = criterion(outputs, price)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")

# --- Evaluation ---
model.eval()
all_preds = []
all_actuals = []

with torch.no_grad():
    for tab, img, price in test_loader:
        tab, img = tab.to('cuda'), img.to('cuda')
        preds = model(tab, img)
        all_preds.extend(preds.cpu().numpy())
        all_actuals.extend(price.numpy())

# Calculate Task Requirements: MAE and RMSE
mae = mean_absolute_error(all_actuals, all_preds)
rmse = np.sqrt(mean_squared_error(all_actuals, all_preds))

print("-" * 30)
print(f"FINAL METRICS:")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print("-" * 30)

Starting Training...
Epoch 1 | Loss: 96.0736
Epoch 2 | Loss: 1.1244
Epoch 3 | Loss: 1.0027
Epoch 4 | Loss: 1.0020
Epoch 5 | Loss: 0.9896
------------------------------
FINAL METRICS:
Mean Absolute Error (MAE): 0.8278
Root Mean Squared Error (RMSE): 0.9784
------------------------------
